# UniChart — Bar `overlay_sets`: other datasets' values as markers

A bar chart draws one bar per selected dataset. Sometimes you want to compare the
bars against **the same variable from other datasets**, without drawing more bars.
Examples: a test run against its model prediction, or each engine against the fleet
average.

`markers=` can't do this, because it overlays **columns** of the bar's own dataset.
`overlay_sets=` overlays **datasets**: each overlay set is drawn as one marker per
category instead of as bars.

| call | result |
|---|---|
| `overlay_sets=[1, 4]` | sets 1 and 4 are drawn as markers at each category's center |
| `overlay_sets={0: 1, 2: 3}` | set 1's markers sit over set 0's bars, set 3's over set 2's |
| `overlay_sets={0: [1, 4], 2: [3, 4]}` | set 4 is drawn over both bar sets |

Each marker is the value the overlay set's bar would have had: the same `agg`
over that set's own rows. Hover shows that set's own row count.

Covered:

- **A.** One bar set with overlay sets.
- **B.** Overlay sets keep their own style (color, marker, size, fill).
- **C.** Several bar sets: unpaired overlay sets sit at the category center.
- **D.** Pairing overlay sets with particular bar sets using a dict.
- **E.** `by='sets'`: overlays on every panel, or only on their paired panels.
- **F.** `agg` and hover counts come from each overlay set's own rows.
- **G.** Selection rules: overlay sets are never bars, selected or not.
- **H.** Guard rails: what raises and what warns.
- **I.** Sessions save and replay the pairing.

Every section ends with `check(...)` assertions, so running the notebook top to
bottom also tests the feature.

## Setup

In [ ]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import contextlib, io, tempfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from unichart import UnichartNotebook

PASSED = 0
def check(cond, msg):
    global PASSED
    assert cond, 'FAILED: ' + msg
    PASSED += 1
    print('PASS:', msg)

# Per-phase engine data. Tests have 12 samples per phase; the models have 3
# (so hover counts differ) and sit close to their engine's test values.
PHASES = ['TAXI', 'TAKEOFF', 'CLIMB', 'CRUISE', 'DESCENT']
BASE_EGT = dict(TAXI=430, TAKEOFF=880, CLIMB=810, CRUISE=700, DESCENT=520)
BASE_FF = dict(TAXI=300, TAKEOFF=2600, CLIMB=2100, CRUISE=1300, DESCENT=500)
rng = np.random.default_rng(7)

def engine(egt_shift, ff_scale, n_per_phase, noise):
    rows = [{'PHASE': p,
             'EGT': BASE_EGT[p] + egt_shift + rng.normal(0, noise),
             'FF': BASE_FF[p] * ff_scale * (1 + rng.normal(0, noise / 1000))}
            for p in PHASES for _ in range(n_per_phase)]
    return pd.DataFrame(rows)

SETS = [
    ('Engine A test',  engine(+40, 1.04, 12, 12)),   # set 0
    ('Engine A model', engine(+25, 1.02,  3,  2)),   # set 1
    ('Engine B test',  engine(-40, 0.96, 12, 12)),   # set 2
    ('Engine B model', engine(-55, 0.97,  3,  2)),   # set 3
    ('Fleet average',  engine(  0, 1.00, 40, 15)),   # set 4
]

def build():
    # A fresh notebook holding the five sets. Plots come back as PNGs.
    with contextlib.redirect_stdout(io.StringIO()):
        uc = UnichartNotebook()
        uc.set_copy_buttons(False)
        for title, df in SETS:
            uc.load_df(df, title=title)
    uc.set_static_images(True, scale=1)
    uc.figsize = (10, 4.5)
    return uc

def traces(uc, kind):
    return [t for t in uc.last_fig.data if t.type == kind]

def names(uc, kind):
    return [t.name for t in traces(uc, kind)]

def phase_stat(uc, idx, col, how='mean'):
    # What the overlay marker should show: set idx's own per-phase agg.
    return uc.sets[idx].cols(['PHASE', col]).groupby('PHASE')[col].agg(how).to_dict()

def close(a, b):
    return a.keys() == b.keys() and all(abs(a[k] - b[k]) < 1e-9 for k in a)

uc = build()
[d.title_format for d in uc.sets]

---
## A. One bar set with overlay sets

Only Engine A's test is selected, so it is the only set drawn as bars. The model
(set 1) and the fleet average (set 4) are drawn as markers. Overlay sets are drawn
**whether or not they are selected**, so there's no need to select them.

In [ ]:
uc = build()
uc.select(0)
img = uc.bar(x='PHASE', y='EGT', overlay_sets=[1, 4],
             suptitle='A. Engine A test vs. its model and the fleet average')

check(names(uc, 'bar') == ['0: Engine A test'], 'only set 0 draws bars')
check(names(uc, 'scatter') == ['1: Engine A model', '4: Fleet average'],
      'sets 1 and 4 draw one marker trace each')
for t, idx in zip(traces(uc, 'scatter'), (1, 4)):
    check(close(dict(zip(t.x, t.y)), phase_stat(uc, idx, 'EGT')),
          f'set {idx} markers are its own per-phase mean')
img

---
## B. Overlay sets keep their own style

Each overlay set is drawn with its own `color`, `marker`, `markersize`, `fill` and
`alpha`, so the usual per-set setters apply. Here the model is a hollow diamond and the
fleet average a large black cross.

If a set has markers turned off (`marker('none')` or `markersize(..., 0)`), its
overlay falls back to the set's default symbol and size. A marker is the only thing
an overlay set draws, so turning it off would hide the set completely.

In [ ]:
uc = build()
uc.select(0)
uc.marker(1, 'D')
uc.fill(1, False)
uc.marker(4, 'X')
uc.color(4, 'black')
uc.markersize(4, 16)
img = uc.bar(x='PHASE', y='EGT', overlay_sets=[1, 4],
             suptitle='B. Overlay sets use their own set style')

model, fleet = traces(uc, 'scatter')
check(model.marker.symbol == 'diamond', 'set 1 uses its own marker')
check(model.marker.color == 'rgba(0,0,0,0)' and model.marker.line.color == uc.sets[1].color,
      'fill(False) makes set 1 hollow, outlined in its color')
check(fleet.marker.color == 'black' and fleet.marker.size == 16,
      "set 4 uses its own color and size")
img

---
## C. Several bar sets: unpaired overlays sit at the category center

With both engines' tests drawn as bars, an overlay set given as a plain selector
isn't tied to either bar. Its marker sits at the **center of the category**, between
the bars. That suits a shared reference like the fleet average. With several `y`
variables, each panel gets its own markers.

In [ ]:
uc = build()
uc.select([0, 2])
uc.marker(4, 'X')
uc.color(4, 'black')
uc.markersize(4, 14)
img = uc.bar(x='PHASE', y=['EGT', 'FF'], overlay_sets=4, ncols=2,
             suptitle='C. Two engines vs. the fleet average (category center)')

marks = traces(uc, 'scatter')
check(len(marks) == 2, 'one overlay trace per y panel')
check({t.offsetgroup for t in marks} == {'overlay_sets'},
      'unpaired overlays use their own alignment group (category center)')
check(sum(t.showlegend for t in marks) == 1, 'one legend entry for the overlay set')
img

---
## D. Pairing overlay sets with particular bar sets

Each model belongs to one engine, so a marker between the bars doesn't say which
engine it predicts. A **dict** pairs them. It maps each bar set to the overlay set(s)
drawn on it, `{bar set: overlay set(s)}`, and both sides take the usual selector:

In [ ]:
uc = build()
uc.select([0, 2])
uc.marker([1, 3], 'D')
img = uc.bar(x='PHASE', y='EGT', overlay_sets={0: 1, 2: 3},
             suptitle='D1. Each engine with its own model')

groups = {t.name: t.offsetgroup for t in traces(uc, 'scatter')}
check(groups == {'1: Engine A model': 'set_0', '3: Engine B model': 'set_2'},
      "set 1 rides set 0's bars, set 3 rides set 2's")
img

An overlay set can be paired with **several** bar sets. It's drawn over each of
them and still gets a single legend entry. Here the fleet average is drawn over both
engines, next to each engine's model:

In [ ]:
uc = build()
uc.select([0, 2])
uc.marker([1, 3], 'D')
uc.marker(4, 'X')
uc.color(4, 'black')
uc.markersize(4, 14)
img = uc.bar(x='PHASE', y='EGT', overlay_sets={0: [1, 4], 2: [3, 4]},
             suptitle='D2. Each engine with its model and the fleet average')

fleet = [t for t in traces(uc, 'scatter') if t.name == '4: Fleet average']
check(sorted(t.offsetgroup for t in fleet) == ['set_0', 'set_2'],
      'set 4 is drawn over both bar sets')
check(sum(t.showlegend for t in fleet) == 1, '...with a single legend entry')
img

Bar sets left out of the dict get no markers. To mix paired and centered markers,
pair the shared set with every bar set, as above.

---
## E. `by='sets'`: one panel per bar set

In the per-dataset view, overlay sets get **no panel of their own**. An unpaired
overlay set appears on **every** panel, over the matching variable's bar:

In [ ]:
uc = build()
uc.select([0, 2])
uc.marker(4, 'X')
uc.color(4, 'black')
uc.markersize(4, 14)
img = uc.bar(x='PHASE', y='EGT', by='sets', overlay_sets=4,
             suptitle='E1. by=sets, fleet average on every panel')

check(len(uc.last_fig.layout.annotations) >= 2 and
      {a.text for a in uc.last_fig.layout.annotations} >= {'0: Engine A test', '2: Engine B test'},
      'one panel per bar set (none for the overlay set)')
check({t.xaxis for t in traces(uc, 'scatter')} == {'x', 'x2'}, 'the fleet average is on both panels')
img

A paired overlay set appears only on its own bar sets' panels:

In [ ]:
uc = build()
uc.select([0, 2])
uc.marker([1, 3], 'D')
img = uc.bar(x='PHASE', y='EGT', by='sets', overlay_sets={0: 1, 2: 3},
             suptitle='E2. by=sets, each model on its own engine')

check({(t.name, t.xaxis) for t in traces(uc, 'scatter')} ==
      {('1: Engine A model', 'x'), ('3: Engine B model', 'x2')},
      "each model is drawn only on its engine's panel")
img

---
## F. `agg` and hover counts come from the overlay set's own rows

The marker is the overlay set's **own** aggregate, using the same `agg` as the bars,
so any reducer works, `sum` and `count` included. Hover reports
`<agg> EGT: <value> (n=<rows>)`, where `n` counts that set's own rows: 3 per phase for
a model and 12 for a test. Static images don't show hover, so the checks read the
counts from the figure instead.

In [ ]:
uc = build()
uc.select(0)
img = uc.bar(x='PHASE', y='EGT', agg='max', overlay_sets={0: 1},
             suptitle="F. agg='max': each set's own maximum")

bar, = traces(uc, 'bar')
model, = traces(uc, 'scatter')
check(close(dict(zip(bar.x, bar.y)), phase_stat(uc, 0, 'EGT', 'max')), 'bars are set 0 max')
check(close(dict(zip(model.x, model.y)), phase_stat(uc, 1, 'EGT', 'max')), 'markers are set 1 max')
check(set(bar.customdata) == {12} and set(model.customdata) == {3},
      'hover n is each set\'s own row count (12 vs 3)')
print('hover template:', model.hovertemplate)
img

---
## G. Selection rules

- An overlay set is **never** drawn as bars, even when it's selected.
- An overlay set is drawn even when it **isn't** selected.
- Every other selected set is drawn as bars, as usual.

With everything selected, the two models become markers, while the fleet average
(not in the dict) stays a bar:

In [ ]:
uc = build()
uc.select('all')
uc.marker([1, 3], 'D')
img = uc.bar(x='PHASE', y='EGT', overlay_sets={0: 1, 2: 3},
             suptitle='G. All selected: models become markers, the rest stay bars')

check(names(uc, 'bar') == ['0: Engine A test', '2: Engine B test', '4: Fleet average'],
      'selected overlay sets are not drawn as bars')
img

---
## H. Guard rails

A pairing that can't be drawn raises a `ValueError` saying which pair is wrong.
`by='dataset_x'`, where the datasets themselves are the categories, ignores
`overlay_sets` with a warning.

In [ ]:
uc = build()
uc.select([0, 2])

def error_of(y='EGT', **kw):
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            uc.bar(x='PHASE', y=y, **kw)
    except ValueError as e:
        return str(e)

cases = {
    'pair with an unselected set':   dict(overlay_sets={4: 1}),
    'a set as both bar and overlay': dict(overlay_sets={0: 2, 2: 3}),
    'no such dataset':               dict(overlay_sets={0: 99}),
    'column no overlay set has':     dict(overlay_sets=1, y='NOT_A_COLUMN'),
}
for label, kw in cases.items():
    msg = error_of(**kw)
    print(f'{label:32s} -> {msg}')
    check(msg is not None, f'{label} raises ValueError')

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    with contextlib.redirect_stdout(io.StringIO()):
        uc.bar(y='EGT', by='dataset_x', overlay_sets=1)
check(any('overlay_sets' in str(w.message) for w in caught), "by='dataset_x' warns it ignores overlay_sets")

out = io.StringIO()
uc.select(1)
with contextlib.redirect_stdout(out):
    result = uc.bar(x='PHASE', y='EGT', overlay_sets=1)
print(out.getvalue().strip())
check(result is None, 'nothing is drawn when every selected set is an overlay set')

---
## I. Sessions save and replay the pairing

The plot call is recorded with the session. JSON turns dict keys into strings
(`{"0": 1}`), and a string key is read as a set selector, so the replayed chart
matches the original.

In [ ]:
uc = build()
uc.select([0, 2])
uc.marker([1, 3], 'D')
with contextlib.redirect_stdout(io.StringIO()):
    uc.bar(x='PHASE', y='EGT', overlay_sets={0: 1, 2: 3})
before = [(t.name, t.offsetgroup) for t in traces(uc, 'scatter')]

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / 'overlay_session.json'
    with contextlib.redirect_stdout(io.StringIO()):
        uc.save_session(path, embed_data='all')
        fresh = UnichartNotebook()
        fresh.set_copy_buttons(False)
        fresh.set_static_images(True, scale=1)
        fresh.load_session(path)

print('recorded call:', uc._last_plot_call)
check([(t.name, t.offsetgroup) for t in traces(fresh, 'scatter')] == before,
      'the replayed chart pairs the same sets')

---
## Summary

In [ ]:
print(f'All {PASSED} checks passed.')